In [1]:
import torch.nn as nn

#==========Model Class===========
#==========Model Class===========
#==========Model Class===========
class FlowPredictor(nn.Module):
    def __init__(self, hidden_dim, output_dim, dropout, input_dim=1):
        super(FlowPredictor, self).__init__()
        #Layer1
        self.lstm1 = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.dropout1 = nn.Dropout(dropout)
        
        #Layer2
        self.lstm2 = nn.LSTM(hidden_dim, hidden_dim // 2, batch_first=True)
        self.dropout2 = nn.Dropout(dropout)
        
        #Layer3
        self.lstm3 = nn.LSTM(hidden_dim // 2, hidden_dim // 4, batch_first=True)
        self.dropout3 = nn.Dropout(dropout)

        #Last layer
        self.fc = nn.Linear(hidden_dim // 4, output_dim)
        #self.fc = nn.Linear(hidden_dim // 2, output_dim)

    def forward(self, x):
        # x shape: (batch, seq_len, input_dim)
        lstm1_out, _ = self.lstm1(x)
        out = self.dropout1(lstm1_out)

        lstm2_out, _ = self.lstm2(lstm1_out)
        out = self.dropout2(lstm2_out)

        lstm3_out, _ = self.lstm3(lstm2_out)

        # Use the last hidden state for prediction
        last_hidden = lstm3_out[:, -1, :]
        out = self.dropout3(last_hidden)
        #last_hidden = lstm2_out[:, -1, :]
        #out = self.dropout2(last_hidden)
        out = self.fc(out)
        return out

In [2]:
import torch
import joblib
import numpy as np
import json
from scipy.signal import savgol_filter
#=========Real Service========
#=========Real Service========
#=========Real Service========
class ReservoirInferenceService:
    def __init__(self, reservoir_configs, window_size=180):
        self.window_size = window_size
        
        # reservoir_configs = {'g': {'weights': 'path', 'scaler': 'path'}, 'i': {...}}
        self.models = {}
        self.scalers = {}

        for name, paths in reservoir_configs.items():
            # Load Config (JSON)
            with open(paths['config'], 'r') as f:
                config = json.load(f)

            # Load Scaler
            self.scalers[name] = joblib.load(paths['scaler'])
            
            # Load Model
            model = FlowPredictor(
                hidden_dim=config['units'],
                output_dim=config['forecast_size'],
                dropout=config['dropout']
            )
            model.load_state_dict(torch.load(paths['weights'], map_location=torch.device('cpu')))
            model.eval()
            self.models[name] = model

    def predict(self, reservoir_name, raw_data):
        # 0. Select the correct tools
        model = self.models[reservoir_name]
        scaler = self.scalers[reservoir_name]
        
        # 1. Scale
        n_min = self.window_size #input window size
        filtered_data = savgol_filter(raw_data, window_length=31, polyorder=1)
        scaled_data = scaler.transform(filtered_data.reshape(-1, 1))
        input_tensor = torch.FloatTensor(scaled_data).view(1, n_min, 1)
        
        # 2. Predict
        with torch.no_grad():
            prediction = model(input_tensor)
            
        # 3. Inverse Scale
        return scaler.inverse_transform(prediction.numpy())


**generator.py**

In [3]:
import time
from datetime import timedelta
#from inference import ReservoirInferenceService
from redis import Redis
import pymysql
import pandas as pd

# 1. SETUP
# Initialize Inference 
window_size=180
curdate = '2024-01-01 00:01'
configs = {
    "g": {"weights": "../model/g_resv/g_resv_flow_model.pth", 
          "scaler": "../model/g_resv/g_resv_scaler.pkl",
          "config": "../model/g_resv/g_resv_config.json"
          }
}
service = ReservoirInferenceService(configs,window_size=window_size)
redis_client = Redis(host="10.125.121.184", port=6379, decode_responses=True)

def connect_MySQL():
    return pymysql.connect(
        host='10.125.121.184',
        port=3306,
        user='musthave',
        password='tiger',
        database='pms_db_dev_gs',
        charset='utf8mb4',
    )

In [4]:
def get_latest_window(resv, start_date):
    resv_dict = {
        'a':4,
        'b':5,
        'c':6,
        'd':7,
        'e':8,
        'f':9,
        'g':10,
        'h':11,
        'i':12,
        'j':13,
        'k':14,
        'l':15
    }
    connection = connect_MySQL()
    try: 
        query = """
            SELECT collected_at, flow_out
            FROM reservoir_minutely 
            WHERE facility_id = %s 
            AND collected_at >= %s
            AND collected_at < DATE_ADD(%s, INTERVAL 3 HOUR)
            ORDER BY collected_at ASC 
        """
        facility_id = resv_dict[resv]
        df = pd.read_sql(query, connection, params=(facility_id, start_date, start_date))
        return df['flow_out'].values, df['collected_at'].values[-1]
    finally:
        connection.close()

In [5]:
def format_to_json(prediction, last_input_time):
    last_input_time = pd.to_datetime(last_input_time)
    start_pred_time = last_input_time + timedelta(minutes=1)
    prediction = prediction.reshape(-1,)

    forecast_time = pd.date_range(
        start=start_pred_time,
        periods=len(prediction),
        freq='1min'
        )
    
    # print(f'prediction : {prediction}')
    # print(f'last_input_time : {last_input_time}')
    # print(f'forecast_time : {forecast_time}')

    # print(f'prediction : {prediction.shape}')
    # print(f'forecast : {forecast_time.shape}')

    pred_df = pd.DataFrame({
        'time': forecast_time,
        'resv_flow_pred':prediction
    })

    # 1. Convert the DataFrame rows to a list of dictionaries
    json_data = pred_df.to_json(orient='records', date_format='iso')
    # 2. Parse that string back to a Python object and wrap it
    # (This step is necessary if you want the 'predictions' header)
    final_dict = {
        "resv_flow_pred": json.loads(json_data)
    }

    # 3. Convert the final dictionary to a JSON string for Redis
    json_for_redis = json.dumps(final_dict)
    date_for_redis = json.dumps(str(last_input_time))
    print(date_for_redis)
    return json_for_redis, date_for_redis

# 2. THE LOOP
def run_generator():
    print("Generator started. Monitoring reservoir flow...")
    while True:
        try:
            input_window, last_input_time= get_latest_window('g', start_date=curdate)
            prediction = service.predict('g', input_window)
            
            json_pred, json_date = format_to_json(prediction, last_input_time)
            redis_client.set(f"{'g'}_prediction", json_pred)
            redis_client.set(f"{'g'}_last_updated", json_date)

            print(f"Cycle complete at {time.ctime()}")
            time.sleep(60*10) # Wait for the next minute
            
        except Exception as e:
            print(f"Error: {e}")
            time.sleep(10) # Short wait before retry

In [ ]:
run_generator()

Generator started. Monitoring reservoir flow...
"2024-01-01 03:00:00"
Cycle complete at Thu Feb  5 15:10:41 2026


C:\Users\user\AppData\Local\Temp\ipykernel_2916\2974686163.py:27: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, connection, params=(facility_id, start_date, start_date))
c:\Users\user\miniconda3\envs\mypy\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
